# LeetCode 3981 — Count Distinct Ways to Form Target from Two Strings bằng Python

- Link: https://leetcode.com/problems/count-distinct-ways-to-form-target-from-two-strings/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP), đếm dãy con (subsequence), chuỗi, tổng tiền tố (prefix sum), modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def interleaveCharacters(self, word1: str, word2: str, target: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bạn có hai chuỗi `word1`, `word2` và một chuỗi đích `target`. Với **mỗi ký tự của `target`** ta phải chọn một ký tự trùng khớp, nằm trong `word1` **hoặc** `word2`. Các chỉ số chọn trong `word1` phải tăng ngặt, các chỉ số chọn trong `word2` cũng tăng ngặt, và phải dùng **ít nhất một ký tự từ mỗi chuỗi**. Hỏi có bao nhiêu cách (chia lấy dư cho `10^9 + 7`).

Đây là bài "đếm số dãy con" quen thuộc (như *Distinct Subsequences*), nhưng nâng cấp: có **hai** chuỗi nguồn cùng làm việc, nên trạng thái phải nhớ **hai con trỏ**. Điểm thú vị là cách xử lý điều kiện "ít nhất một ký tự từ mỗi chuỗi" mà không cần thêm cờ phức tạp.

Sau notebook này bạn sẽ:

1. Hiểu "chỉ số tăng ngặt" và cách đếm số dãy con bằng DP.
2. Biết chọn **trạng thái** cho bài toán hai con trỏ: `(đã khớp bao nhiêu ký tự target, vị trí cuối đã dùng trong word1, vị trí cuối đã dùng trong word2)`.
3. Biết dùng **tổng tiền tố** để giảm một vòng lặp lồng ra khỏi bước chuyển trạng thái (từ `O(n)` xuống `O(1)`).
4. Viết 4 lời giải (vét cạn, DP chậm, DP tổng tiền tố, DP tiền tố + bù trừ) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích độ phức tạp `O(|target| · |word1| · |word2|)` và vì sao nó đủ nhanh với giới hạn 100.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `word1`, `word2`, `target` (chữ thường). Một **cách** là một danh sách `|target|` lựa chọn; lựa chọn thứ `k` là một cặp `(nguồn, chỉ số)` với:

- nguồn là `word1` hoặc `word2`;
- ký tự tại chỉ số đó **bằng** `target[k]`;
- trong các lựa chọn từ `word1` (đọc theo thứ tự `k` tăng dần) chỉ số **tăng ngặt**; tương tự với `word2`;
- cả hai nguồn đều được dùng ít nhất một lần.

Hai cách khác nhau nếu có ít nhất một vị trí `k` mà nguồn hoặc chỉ số khác nhau. Trả về số cách `mod 10^9 + 7`.

**Ví dụ 1:** `word1 = "abc"`, `word2 = "bac"`, `target = "abc"` → `5`.

**Ví dụ 2:** `word1 = "cd"`, `word2 = "cd"`, `target = "ccd"` → `4`.

**Ví dụ 3:** `word1 = "xy"`, `word2 = "xy"`, `target = "xyxy"` → `2`.

**Ví dụ 4:** `word1 = "ab"`, `word2 = "cde"`, `target = "ace"` → `1`.

**Ràng buộc**

- `1 <= word1.length, word2.length, target.length <= 100`
- Chỉ gồm chữ cái thường tiếng Anh.

Lưu ý cách hiểu: thứ tự lấy ký tự là thứ tự của `target`. Ta **không** cần lấy các ký tự của `word1` và `word2` theo kiểu "xen kẽ cố định"; mỗi ký tự `target[k]` tự quyết định nó lấy từ chuỗi nào.

In [1]:
examples = [
    (("abc", "bac", "abc"), 5),
    (("cd", "cd", "ccd"), 4),
    (("xy", "xy", "xyxy"), 2),
    (("ab", "cde", "ace"), 1),
]
for (w1, w2, t), expected in examples:
    print(f"word1={w1!r:6} word2={w2!r:6} target={t!r:7} -> mong đợi {expected}")

word1='abc'  word2='bac'  target='abc'   -> mong đợi 5
word1='cd'   word2='cd'   target='ccd'   -> mong đợi 4
word1='xy'   word2='xy'   target='xyxy'  -> mong đợi 2
word1='ab'   word2='cde'  target='ace'   -> mong đợi 1


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và "chỉ số tăng ngặt"

Một **dãy con** của chuỗi `s` là chuỗi nhận được bằng cách chọn vài ký tự theo **thứ tự xuất hiện** (không nhất thiết liền nhau). "Chỉ số tăng ngặt" nghĩa là `i1 < i2 < i3 < ...`.

Quan trọng: hai cách chọn **khác chỉ số** là hai cách khác nhau dù ký tự giống nhau. Ví dụ `"aa"`: chọn `a` ở chỉ số 0 hay chỉ số 1 là hai cách khác nhau.

In [2]:
from itertools import combinations

s = "abab"
# liệt kê mọi bộ chỉ số tăng ngặt có độ dài 2 và ký tự tạo thành
for idx in combinations(range(len(s)), 2):
    print(idx, "->", "".join(s[i] for i in idx))
# đếm số cách chọn chỉ số để được "ab"
cnt = sum(1 for idx in combinations(range(len(s)), 2) if "".join(s[i] for i in idx) == "ab")
print("số cách tạo 'ab' từ 'abab':", cnt)

(0, 1) -> ab
(0, 2) -> aa
(0, 3) -> ab
(1, 2) -> ba
(1, 3) -> bb
(2, 3) -> ab
số cách tạo 'ab' từ 'abab': 3


### 3.2. Bảng 2 chiều / 3 chiều trong Python và cái bẫy `[[0]*m]*n`

DP cần bảng nhiều chiều. Cách tạo đúng:

```python
dp = [[0] * (m + 1) for _ in range(n + 1)]
```

Cách **sai**: `[[0] * (m + 1)] * (n + 1)` — nó lặp lại **cùng một** list `n + 1` lần, sửa một dòng là sửa tất cả.

In [3]:
bad = [[0] * 3] * 2
bad[0][0] = 7
print("sai :", bad)          # cả hai dòng đều đổi

good = [[0] * 3 for _ in range(2)]
good[0][0] = 7
print("đúng:", good)

sai : [[7, 0, 0], [7, 0, 0]]
đúng: [[7, 0, 0], [0, 0, 0]]


### 3.3. Modulo `10^9 + 7`

Số cách có thể cực lớn nên đề yêu cầu `mod`. Quy tắc: cộng và nhân đều có thể lấy `mod` **sau từng bước**; riêng phép trừ trong Python `(a - b) % MOD` luôn ra số không âm (khác với một số ngôn ngữ khác).

In [4]:
MOD = 10**9 + 7
a, b = 5, 9
print((a - b) % MOD)                # 1000000003, không phải -4
print((10**18) % MOD, ((10**9) % MOD) * ((10**9) % MOD) % MOD)   # hai kết quả giống nhau

1000000003
49 49


### 3.4. Tổng tiền tố (prefix sum / running sum)

Nếu cần nhiều lần tính `x[0] + x[1] + ... + x[i]`, ta cộng dồn một lần: `run += x[i]`, rồi dùng `run` ngay. Đây là chìa khóa để biến "vòng lặp lồng" thành `O(1)` ở phần 7.

In [5]:
x = [3, 1, 4, 1, 5]
run = 0
prefix = []
for v in x:
    run += v
    prefix.append(run)
print(prefix)          # [3, 4, 8, 9, 14]
print("tổng x[0..2] =", prefix[2])

[3, 4, 8, 9, 14]
tổng x[0..2] = 8


## 4. Ý tưởng nền tảng / trực giác

Mỗi ký tự của `target` được "gán" cho `word1` hoặc `word2`. Nếu ta đã quyết định tập vị trí `S1` của `target` đi vào `word1`, thì phần còn lại `S2` đi vào `word2`. Khi đó:

> số cách = (số cách `target[S1]` là dãy con của `word1`) × (số cách `target[S2]` là dãy con của `word2`)

rồi cộng lại theo mọi cách chọn `S1` khác rỗng, `S2` khác rỗng.

**Tính tay ví dụ 3** (`word1 = word2 = "xy"`, `target = "xyxy"`): 

- `S1 = {0,1}` (chữ `"xy"`), `S2 = {2,3}` (chữ `"xy"`): `1 × 1 = 1`.
- `S1 = {2,3}`, `S2 = {0,1}`: `1 × 1 = 1`.
- `S1 = {0,3}` (chữ `"xy"`), `S2 = {1,2}` (chữ `"yx"`): `"yx"` không là dãy con của `"xy"` → `0`.
- các cách chia khác đều có một vế không thể thành dãy con của `"xy"` (độ dài > 2 hoặc sai thứ tự).

Tổng = `2`. ✓

Cách làm thẳng thừng này duyệt `2^|target|` cách chia nên chỉ dùng được cho `target` rất ngắn. Ta kiểm chứng ý tưởng bằng code nhỏ.

In [6]:
from itertools import combinations

def count_subseq(word, pattern):
    """Số cách chọn chỉ số tăng ngặt trong word để tạo thành pattern (đệ quy, chậm, chỉ để demo)."""
    if not pattern:
        return 1
    if not word:
        return 0
    res = count_subseq(word[1:], pattern)                 # bỏ ký tự đầu của word
    if word[0] == pattern[0]:
        res += count_subseq(word[1:], pattern[1:])        # dùng ký tự đầu
    return res

def split_count(word1, word2, target):
    T = len(target)
    total = 0
    for mask in range(1, (1 << T) - 1):                   # S1 khác rỗng, S2 khác rỗng
        p1 = "".join(target[k] for k in range(T) if mask >> k & 1)
        p2 = "".join(target[k] for k in range(T) if not (mask >> k & 1))
        total += count_subseq(word1, p1) * count_subseq(word2, p2)
    return total

for (w1, w2, t), expected in examples:
    print(split_count(w1, w2, t), expected)

5 5
4 4
2 2
1 1


## 5. Cách 1 — Vét cạn: thử mọi lựa chọn từng ký tự của `target`

**Ý tưởng.** Đệ quy `go(k, p1, p2, dùng1, dùng2)`:

- `k`: đã chọn xong `k` ký tự đầu của `target`;
- `p1`, `p2`: chỉ số nhỏ nhất còn được phép dùng trong `word1`, `word2`;
- hai cờ `dùng1`, `dùng2`: đã dùng nguồn đó chưa.

Với `target[k]`, thử mọi chỉ số `i >= p1` của `word1` có ký tự bằng nó (rồi đệ quy với `p1 = i + 1`), và mọi chỉ số `j >= p2` của `word2`. Khi `k == |target|` ta trả `1` nếu cả hai cờ đều bật, ngược lại `0`.

In [7]:
MOD = 10**9 + 7

def brute_force(word1, word2, target):
    n, m, T = len(word1), len(word2), len(target)

    def go(k, p1, p2, used1, used2):
        if k == T:
            return 1 if (used1 and used2) else 0
        ch = target[k]
        total = 0
        for i in range(p1, n):
            if word1[i] == ch:
                total += go(k + 1, i + 1, p2, True, used2)
        for j in range(p2, m):
            if word2[j] == ch:
                total += go(k + 1, p1, j + 1, used1, True)
        return total

    return go(0, 0, 0, False, False) % MOD

for (w1, w2, t), expected in examples:
    got = brute_force(w1, w2, t)
    print(got, expected)
    assert got == expected

5 5
4 4
2 2
1 1


**Độ phức tạp.** Mỗi ký tự có thể có tới `|word1| + |word2|` lựa chọn nên thời gian là **mũ**, cỡ `O((n + m)^T)` trong trường hợp xấu. Chỉ dùng cho input rất nhỏ (và để đối chiếu).

**Nút thắt.** Rất nhiều nhánh đệ quy rơi về cùng một trạng thái `(k, p1, p2, cờ)`. Ví dụ hai cách chọn khác nhau ở các ký tự đầu nhưng cùng kết thúc với `p1 = 3`, `p2 = 2` sẽ cùng đếm lại phần đuôi giống hệt. Đó là dấu hiệu của **DP**: ghi nhớ kết quả theo trạng thái.

## 6. Bước đột phá

**Insight 1 — trạng thái nhỏ gọn.** Điều duy nhất ảnh hưởng tới tương lai là: đã khớp bao nhiêu ký tự `k`, **chỉ số cuối cùng** đã dùng trong `word1` (gọi `a`) và trong `word2` (gọi `b`). Ta mã hóa `a = 0` nếu chưa dùng `word1`, và `a = i + 1` nếu chỉ số cuối đã dùng là `i`. Tương tự `b`.

Nhờ cách mã hóa này, **điều kiện "dùng cả hai nguồn" chỉ còn là `a >= 1` và `b >= 1` ở cuối** — không cần hai cờ riêng.

**Insight 2 — chuyển trạng thái.** Từ trạng thái `(k, a, b)` với ký tự cần khớp `ch = target[k]`:

- lấy từ `word1` tại chỉ số `i` bất kỳ với `i >= a` và `word1[i] == ch` → `(k+1, i+1, b)`;
- lấy từ `word2` tại chỉ số `j >= b` với `word2[j] == ch` → `(k+1, a, j+1)`.

**Insight 3 — tổng tiền tố.** Trạng thái đích `(k+1, i+1, b)` nhận từ **mọi** `a <= i`:

`new[i+1][b] += cur[0][b] + cur[1][b] + ... + cur[i][b]`

Đó là một tổng tiền tố theo `a`, tính dần được bằng một biến `run`. Mỗi trạng thái đích chỉ tốn `O(1)`.

**Quy tắc "nếu… thì…":**

- Nếu `word1[i] == target[k]` thì `new[i+1][b] += (tổng cur[a'][b] với a' <= i)` với mọi `b`.
- Nếu `word2[j] == target[k]` thì `new[a][j+1] += (tổng cur[a][b'] với b' <= j)` với mọi `a`.
- Đáp án = tổng `cur[a][b]` với `a >= 1` và `b >= 1` sau khi xử lý hết `target`.

In [8]:
# Mã hóa trạng thái: a = 0 (chưa dùng word1) hoặc a = i + 1 (chỉ số cuối đã dùng là i)
word1 = "ab"
for a in range(len(word1) + 1):
    if a == 0:
        print("a = 0 : chưa dùng ký tự nào của word1; ký tự tiếp theo có thể ở chỉ số >= 0")
    else:
        print(f"a = {a} : chỉ số cuối đã dùng là {a-1} ({word1[a-1]!r}); ký tự tiếp theo ở chỉ số >= {a}")

a = 0 : chưa dùng ký tự nào của word1; ký tự tiếp theo có thể ở chỉ số >= 0
a = 1 : chỉ số cuối đã dùng là 0 ('a'); ký tự tiếp theo ở chỉ số >= 1
a = 2 : chỉ số cuối đã dùng là 1 ('b'); ký tự tiếp theo ở chỉ số >= 2


## 7. Các cách tiếp theo

Ta giảng 3 lời giải DP, từ dễ hiểu đến gọn.

### 7.1. Cách 2 — DP theo "chỉ số cuối đã dùng", chuyển trạng thái bằng vòng lặp `O(n)`

**Giải thích từng đoạn.**

1. `cur[a][b]` = số cách khớp xong `k` ký tự đầu của `target` mà chỉ số cuối đã dùng là `a` (ở `word1`) và `b` (ở `word2`). Khởi tạo `k = 0`: `cur[0][0] = 1` (chưa dùng gì).
2. Với mỗi ký tự `ch` của `target`, tạo bảng `new` mới toàn `0`.
3. Duyệt mọi `(a, b)` có `cur[a][b] != 0`; thử mọi `i` từ `a` đến `n - 1` để lấy từ `word1`, và mọi `j` từ `b` đến `m - 1` để lấy từ `word2`.
4. Kết quả: cộng `cur[a][b]` với `a >= 1` và `b >= 1`.

In [9]:
def dp_naive(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1
    for ch in target:
        new = [[0] * (m + 1) for _ in range(n + 1)]
        for a in range(n + 1):
            for b in range(m + 1):
                v = cur[a][b]
                if v == 0:
                    continue
                for i in range(a, n):                 # lấy từ word1 tại i >= a
                    if word1[i] == ch:
                        new[i + 1][b] = (new[i + 1][b] + v) % MOD
                for j in range(b, m):                 # lấy từ word2 tại j >= b
                    if word2[j] == ch:
                        new[a][j + 1] = (new[a][j + 1] + v) % MOD
        cur = new
    return sum(cur[a][b] for a in range(1, n + 1) for b in range(1, m + 1)) % MOD

for (w1, w2, t), expected in examples:
    assert dp_naive(w1, w2, t) == expected
print("dp_naive khớp 4 ví dụ")

dp_naive khớp 4 ví dụ


**Độ phức tạp.** Mỗi lớp `k` duyệt `O(n·m)` trạng thái, mỗi trạng thái tốn `O(n + m)` → tổng `O(T · n · m · (n + m))` — với 100 cho mọi độ dài là khoảng `2 · 10^8`, quá chậm trong Python. Cần tổng tiền tố.

### 7.2. Cách 3 — DP với tổng tiền tố (bản chính, `O(T·n·m)`)

**Giải thích từng đoạn.**

- Với lấy từ `word1`: cố định `b`, quét `a` từ `0` đến `n`; giữ `run = cur[0][b] + ... + cur[a][b]`. Tại `a`, nếu `a < n` và `word1[a] == ch`, thì ký tự `word1[a]` được lấy: trạng thái mới `(a + 1, b)` nhận thêm `run` (gồm mọi chỉ số cuối cũ `<= a`).
- Với lấy từ `word2`: đối xứng, cố định `a`, quét `b`.

Hai nguồn cộng vào cùng bảng `new` vì chúng là hai loại lựa chọn khác nhau của cùng một ký tự.

In [10]:
def dp_prefix(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1
    for ch in target:
        new = [[0] * (m + 1) for _ in range(n + 1)]
        # lấy từ word1: cố định b, tổng tiền tố theo a
        for b in range(m + 1):
            run = 0
            for a in range(n + 1):
                run = (run + cur[a][b]) % MOD
                if a < n and word1[a] == ch:
                    new[a + 1][b] = (new[a + 1][b] + run) % MOD
        # lấy từ word2: cố định a, tổng tiền tố theo b
        for a in range(n + 1):
            run = 0
            for b in range(m + 1):
                run = (run + cur[a][b]) % MOD
                if b < m and word2[b] == ch:
                    new[a][b + 1] = (new[a][b + 1] + run) % MOD
        cur = new
    return sum(cur[a][b] for a in range(1, n + 1) for b in range(1, m + 1)) % MOD

for (w1, w2, t), expected in examples:
    assert dp_prefix(w1, w2, t) == expected
print("dp_prefix khớp 4 ví dụ")

dp_prefix khớp 4 ví dụ


**Độ phức tạp.** `O(T · n · m)` thời gian; `O(n · m)` bộ nhớ nhờ chỉ giữ hai lớp `cur`, `new`. Với 100/100/100 là khoảng `2 · 10^6` phép cập nhật.

### 7.3. Cách 4 — DP "tiền tố" cộng bù trừ

Một góc nhìn khác: bỏ điều kiện "dùng cả hai nguồn" trước, đếm **tất cả** cách, rồi trừ đi những cách chỉ dùng một nguồn.

- `g[k][i][j]` = số cách khớp `k` ký tự đầu của `target` chỉ dùng **tiền tố** `word1[:i]` và `word2[:j]` (không bắt buộc dùng hết).
- Ký tự thứ `k` lấy từ `word1` tại chỉ số `i' < i` hoặc từ `word2` tại `j' < j`:
  `g[k][i][j] = Σ_{i' < i, word1[i'] = target[k-1]} g[k-1][i'][j] + Σ_{j' < j, word2[j'] = target[k-1]} g[k-1][i][j']`
- Các tổng này tính dần: `S1[i][j] = S1[i-1][j] + [word1[i-1] = target[k-1]] · g[k-1][i-1][j]`, và tương tự `S2`.
- Cuối cùng: `đáp án = g[T][n][m] − (số dãy con target trong word1) − (số dãy con target trong word2)`. Vì `|target| >= 1` nên không có cách nào "không dùng nguồn nào", không cần cộng bù lại.

Số dãy con `target` trong `word` đếm bằng DP một chiều quen thuộc.

In [11]:
def count_embeddings(word, target):
    """Số cách chọn chỉ số tăng ngặt trong word để tạo target (DP 1 chiều, duyệt ngược)."""
    f = [1] + [0] * len(target)          # f[t] = số cách khớp t ký tự đầu của target
    for c in word:
        for t in range(len(target), 0, -1):
            if target[t - 1] == c:
                f[t] = (f[t] + f[t - 1]) % MOD
    return f[len(target)]

def dp_inclusion_exclusion(word1, word2, target):
    n, m, T = len(word1), len(word2), len(target)
    # g[i][j] của lớp k-1; bắt đầu k = 0: có đúng 1 cách (chọn rỗng) với mọi i, j
    g = [[1] * (m + 1) for _ in range(n + 1)]
    for k in range(1, T + 1):
        ch = target[k - 1]
        ng = [[0] * (m + 1) for _ in range(n + 1)]
        S1 = [[0] * (m + 1) for _ in range(n + 1)]
        S2 = [[0] * (m + 1) for _ in range(n + 1)]
        for i in range(n + 1):
            for j in range(m + 1):
                if i > 0:
                    S1[i][j] = S1[i - 1][j] + (g[i - 1][j] if word1[i - 1] == ch else 0)
                    S1[i][j] %= MOD
                if j > 0:
                    S2[i][j] = S2[i][j - 1] + (g[i][j - 1] if word2[j - 1] == ch else 0)
                    S2[i][j] %= MOD
                ng[i][j] = (S1[i][j] + S2[i][j]) % MOD
        g = ng
    total = g[n][m]
    return (total - count_embeddings(word1, target) - count_embeddings(word2, target)) % MOD

for (w1, w2, t), expected in examples:
    assert dp_inclusion_exclusion(w1, w2, t) == expected
print("dp_inclusion_exclusion khớp 4 ví dụ")

dp_inclusion_exclusion khớp 4 ví dụ


**Độ phức tạp.** `O(T · n · m)` thời gian; bản trên dùng thêm hai bảng `S1`, `S2` nên bộ nhớ `O(n · m)` theo từng lớp `k`. Cách này **dễ sai** hơn cách 3 (phải nhớ trừ hai trường hợp biên), nhưng cho ta một lần đối chiếu độc lập.

## 8. TRACE từng bước

Ta trace `dp_prefix` trên ví dụ 4: `word1 = "ab"`, `word2 = "cde"`, `target = "ace"`. Quy ước: sau mỗi ký tự của `target`, in các trạng thái `(a, b)` có giá trị khác 0, kèm giải nghĩa "chỉ số cuối đã dùng".

Bảng mong đợi:

| Sau ký tự | Trạng thái khác 0 | Ý nghĩa |
|---|---|---|
| (bắt đầu) | `(0,0) = 1` | chưa dùng gì |
| `'a'` | `(1,0) = 1` | `a` lấy từ `word1[0]` (word2 không có `a`) |
| `'c'` | `(1,1) = 1` | `c` lấy từ `word2[0]` (word1 không có `c`) |
| `'e'` | `(1,3) = 1` | `e` lấy từ `word2[2]` |

Cuối cùng chỉ có `(1,3)` thỏa `a >= 1` và `b >= 1` → đáp án `1`.

In [12]:
def trace_dp(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1

    def show(title, tbl):
        items = [(a, b, tbl[a][b]) for a in range(n + 1) for b in range(m + 1) if tbl[a][b]]
        print(title)
        for a, b, v in items:
            la = "chưa dùng" if a == 0 else f"word1[{a-1}]={word1[a-1]!r}"
            lb = "chưa dùng" if b == 0 else f"word2[{b-1}]={word2[b-1]!r}"
            print(f"   (a={a}, b={b}) = {v:>3}   | word1: {la:14} | word2: {lb}")

    show("khởi tạo", cur)
    for k, ch in enumerate(target):
        new = [[0] * (m + 1) for _ in range(n + 1)]
        for b in range(m + 1):
            run = 0
            for a in range(n + 1):
                run += cur[a][b]
                if a < n and word1[a] == ch:
                    new[a + 1][b] += run
        for a in range(n + 1):
            run = 0
            for b in range(m + 1):
                run += cur[a][b]
                if b < m and word2[b] == ch:
                    new[a][b + 1] += run
        cur = new
        show(f"sau khi khớp target[{k}] = {ch!r}", cur)
    ans = sum(cur[a][b] for a in range(1, n + 1) for b in range(1, m + 1))
    print("đáp án (a>=1 và b>=1):", ans)
    return ans

trace_dp("ab", "cde", "ace")

khởi tạo
   (a=0, b=0) =   1   | word1: chưa dùng      | word2: chưa dùng
sau khi khớp target[0] = 'a'
   (a=1, b=0) =   1   | word1: word1[0]='a'   | word2: chưa dùng
sau khi khớp target[1] = 'c'
   (a=1, b=1) =   1   | word1: word1[0]='a'   | word2: word2[0]='c'
sau khi khớp target[2] = 'e'
   (a=1, b=3) =   1   | word1: word1[0]='a'   | word2: word2[2]='e'
đáp án (a>=1 và b>=1): 1


1

In [13]:
# Trace ví dụ 2 để thấy "bắt buộc phải dùng cả hai nguồn": cd / cd / ccd
trace_dp("cd", "cd", "ccd")

khởi tạo
   (a=0, b=0) =   1   | word1: chưa dùng      | word2: chưa dùng
sau khi khớp target[0] = 'c'
   (a=0, b=1) =   1   | word1: chưa dùng      | word2: word2[0]='c'
   (a=1, b=0) =   1   | word1: word1[0]='c'   | word2: chưa dùng
sau khi khớp target[1] = 'c'
   (a=1, b=1) =   2   | word1: word1[0]='c'   | word2: word2[0]='c'
sau khi khớp target[2] = 'd'
   (a=1, b=2) =   2   | word1: word1[0]='c'   | word2: word2[1]='d'
   (a=2, b=1) =   2   | word1: word1[1]='d'   | word2: word2[0]='c'
đáp án (a>=1 và b>=1): 4


4

Ở ví dụ 2, sau hai ký tự `c`, ta thấy các trạng thái như `(1,1)` (một `c` từ mỗi chuỗi) có giá trị `2` (hai thứ tự: `c` đầu từ `word1` rồi `c` sau từ `word2`, hoặc ngược lại). Còn các trạng thái kiểu `(1,0)` đã không thể khớp được ký tự `c` thứ hai (word1 chỉ có một `c` ở chỉ số 0, không thể dùng lại) nên tự biến mất. Ký tự `d` cuối cùng được lấy từ nguồn nào cũng được → nhân đôi thành `4`.

## 9. Code nộp LeetCode

Dùng cách 3 (tổng tiền tố), viết gọn và thay `sum(...)` bằng vòng lặp cho rõ. Lưu ý: tham số tên `word1`, `word2`, `target` — không trùng tên hàm dựng sẵn.

In [14]:
class Solution:
    def interleaveCharacters(self, word1: str, word2: str, target: str) -> int:
        MOD = 10**9 + 7
        n, m = len(word1), len(word2)
        # cur[a][b]: số cách khớp xong một tiền tố của target, chỉ số cuối đã dùng là a-1 (word1), b-1 (word2)
        cur = [[0] * (m + 1) for _ in range(n + 1)]
        cur[0][0] = 1
        for ch in target:
            new = [[0] * (m + 1) for _ in range(n + 1)]
            for b in range(m + 1):                    # lấy ký tự từ word1
                run = 0
                for a in range(n + 1):
                    run += cur[a][b]
                    if a < n and word1[a] == ch:
                        new[a + 1][b] = (new[a + 1][b] + run) % MOD
            for a in range(n + 1):                    # lấy ký tự từ word2
                run = 0
                for b in range(m + 1):
                    run += cur[a][b]
                    if b < m and word2[b] == ch:
                        new[a][b + 1] = (new[a][b + 1] + run) % MOD
            cur = new
        ans = 0
        for a in range(1, n + 1):                     # phải dùng cả hai chuỗi
            for b in range(1, m + 1):
                ans += cur[a][b]
        return ans % MOD


sol = Solution()
for (w1, w2, t), expected in examples:
    got = sol.interleaveCharacters(w1, w2, t)
    print(got, expected)
    assert got == expected

5 5
4 4
2 2
1 1


## 10. Vì sao thuật toán đúng

**Bất biến.** Sau khi xử lý `k` ký tự đầu của `target`, `cur[a][b]` bằng số **danh sách lựa chọn** (mỗi lựa chọn = nguồn + chỉ số) hợp lệ cho `k` ký tự đó, trong đó chỉ số lớn nhất đã dùng ở `word1` là `a - 1` (hoặc `a = 0` nếu chưa dùng) và ở `word2` là `b - 1`.

**Quy nạp.**

- `k = 0`: chỉ có danh sách rỗng, đúng `cur[0][0] = 1`.
- Bước chuyển: một danh sách hợp lệ cho `k + 1` ký tự được tạo duy nhất từ một danh sách hợp lệ cho `k` ký tự cộng thêm lựa chọn cuối `(nguồn, chỉ số)`. Lựa chọn cuối từ `word1` tại `i` hợp lệ khi `word1[i] = ch` và `i` lớn hơn chỉ số `word1` lớn nhất đã dùng, tức `i >= a`. Tổng tiền tố đếm đúng mọi `a <= i`, và mỗi danh sách cũ ứng với đúng một `a`, nên **không đếm trùng, không sót**. Với `word2` đối xứng.
- Tính "chỉ số tăng ngặt trong mỗi chuỗi" được bảo đảm bởi điều kiện `i >= a` (chỉ số mới lớn hơn chỉ số cuối cũ `a - 1`).

**Điều kiện cuối.** Cả hai chuỗi đều được dùng khi và chỉ khi `a >= 1` và `b >= 1`, vì `a = 0` chỉ xảy ra nếu **chưa từng** lấy ký tự nào từ `word1`. Các trạng thái đó được cộng đúng một lần.

**Hai cách khác nhau thật sự khác nhau.** Mỗi danh sách lựa chọn ứng với đúng một đường đi trong DP (vì trạng thái mới `(i+1, b)` hoặc `(a, j+1)` được cộng riêng cho mỗi `i`, `j`), nên số đường đi = số cách theo định nghĩa "khác ở ít nhất một vị trí".

## 11. Phân tích Time/Space

Đặt `n = |word1|`, `m = |word2|`, `T = |target|`.

| Cách | Time | Space |
|---|---|---|
| Vét cạn | `O((n+m)^T)` (mũ) | `O(T)` (ngăn xếp) |
| DP chuyển `O(n+m)` | `O(T·n·m·(n+m))` | `O(n·m)` |
| DP tổng tiền tố (nộp) | `O(T·n·m)` | `O(n·m)` |
| DP tiền tố + bù trừ | `O(T·n·m)` | `O(n·m)` |

Với tối đa `T = n = m = 100`: `T·n·m = 10^6` ô, mỗi ô vài phép cộng → chạy trong khoảng vài trăm mili-giây tới một giây trong Python.

In [15]:
import time, random
rng = random.Random(7)
w1 = "".join(rng.choice("ab") for _ in range(100))
w2 = "".join(rng.choice("ab") for _ in range(100))
t  = "".join(rng.choice("ab") for _ in range(100))
t0 = time.time()
res = Solution().interleaveCharacters(w1, w2, t)
print("kết quả với input 100/100/100:", res, f"- {time.time()-t0:.2f}s")
t0 = time.time()
res2 = dp_inclusion_exclusion(w1, w2, t)
print("cách 4 cho cùng input:", res2, f"- {time.time()-t0:.2f}s")
assert res == res2

kết quả với input 100/100/100: 521643217 - 0.20s


cách 4 cho cùng input: 521643217 - 0.36s


## 12. Bộ test

Chiến lược test:

1. 4 ví dụ của đề (assert).
2. Biên: `target` dài 1 (không thể dùng cả hai chuỗi → `0`); không có ký tự nào khớp; hai chuỗi giống hệt.
3. Phản ví dụ cho việc "quên điều kiện dùng cả hai" (xem phần 13).
4. Đối chiếu vét cạn, `dp_naive`, `dp_prefix`, `dp_inclusion_exclusion`, và `Solution` trên nhiều input ngẫu nhiên nhỏ có seed cố định.

In [16]:
sol = Solution()
# 1) ví dụ đề
for (w1, w2, t), expected in examples:
    assert sol.interleaveCharacters(w1, w2, t) == expected

# 2) biên
assert sol.interleaveCharacters("a", "a", "a") == 0          # chỉ một ký tự: không thể dùng cả hai chuỗi
assert sol.interleaveCharacters("abc", "abc", "z") == 0      # không ký tự nào khớp
assert sol.interleaveCharacters("a", "a", "aa") == 2         # a0 của word1 + a0 của word2, hai thứ tự
assert sol.interleaveCharacters("aa", "a", "aa") == 4        # chọn lại bằng tay: xem Bài tập 2 ở dưới
print("test cố định OK")

test cố định OK


In [17]:
import random
rng = random.Random(2024)
checked = 0
for _ in range(1500):
    n = rng.randint(1, 5); m = rng.randint(1, 5); T = rng.randint(1, 5)
    alpha = rng.choice(["ab", "abc", "a"])
    w1 = "".join(rng.choice(alpha) for _ in range(n))
    w2 = "".join(rng.choice(alpha) for _ in range(m))
    t  = "".join(rng.choice(alpha) for _ in range(T))
    expected = brute_force(w1, w2, t)
    assert dp_naive(w1, w2, t) == expected, (w1, w2, t)
    assert dp_prefix(w1, w2, t) == expected, (w1, w2, t)
    assert dp_inclusion_exclusion(w1, w2, t) == expected, (w1, w2, t)
    assert Solution().interleaveCharacters(w1, w2, t) == expected, (w1, w2, t)
    checked += 1
print("đã đối chiếu", checked, "input ngẫu nhiên: tất cả khớp vét cạn")

đã đối chiếu 1500 input ngẫu nhiên: tất cả khớp vét cạn


In [18]:
# Đối chiếu các cách DP với nhau trên input lớn hơn (vét cạn không còn chạy nổi)
rng = random.Random(99)
for _ in range(30):
    n = rng.randint(10, 40); m = rng.randint(10, 40); T = rng.randint(5, 40)
    alpha = rng.choice(["ab", "abc"])
    w1 = "".join(rng.choice(alpha) for _ in range(n))
    w2 = "".join(rng.choice(alpha) for _ in range(m))
    t  = "".join(rng.choice(alpha) for _ in range(T))
    a1 = dp_prefix(w1, w2, t)
    a2 = dp_inclusion_exclusion(w1, w2, t)
    a3 = Solution().interleaveCharacters(w1, w2, t)
    assert a1 == a2 == a3
print("3 cách DP khớp nhau trên 30 input vừa")

3 cách DP khớp nhau trên 30 input vừa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện "dùng cả hai chuỗi"

**Code sai:** cộng toàn bộ bảng `cur` (kể cả `a = 0` hoặc `b = 0`).
**Vì sao sai:** những trạng thái `a = 0` ứng với cách chỉ lấy ký tự từ `word2` (và ngược lại). Đề bắt buộc dùng cả hai.
**Code đúng:** chỉ cộng `a >= 1` và `b >= 1`.

In [19]:
def wrong_all_states(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1
    for ch in target:
        new = [[0] * (m + 1) for _ in range(n + 1)]
        for b in range(m + 1):
            run = 0
            for a in range(n + 1):
                run += cur[a][b]
                if a < n and word1[a] == ch:
                    new[a + 1][b] += run
        for a in range(n + 1):
            run = 0
            for b in range(m + 1):
                run += cur[a][b]
                if b < m and word2[b] == ch:
                    new[a][b + 1] += run
        cur = new
    return sum(map(sum, cur)) % MOD          # SAI: cộng tất cả trạng thái

print("sai :", wrong_all_states("a", "a", "a"), " đúng:", Solution().interleaveCharacters("a", "a", "a"))      # target 'a': không thể dùng cả hai chuỗi
print("sai :", wrong_all_states("abc", "bac", "abc"), " đúng:", Solution().interleaveCharacters("abc", "bac", "abc"))

sai : 2  đúng: 0
sai : 6  đúng: 5


### Lỗi 2 — Cho phép dùng lại cùng một chỉ số (quên "tăng ngặt")

**Code sai:** khi lấy từ `word1` tại `i`, đặt trạng thái mới là `(i, b)` thay vì `(i + 1, b)`.
**Vì sao sai:** chỉ số cuối đã dùng là `i` nhưng lại được mã hóa thành `a = i` (nghĩa "chỉ số cuối là `i-1`"), nên ký tự thứ `i` được phép dùng lần thứ hai.
**Code đúng:** dùng `new[i + 1][b]` — mã hóa `a = i + 1`.

In [20]:
def wrong_reuse(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1
    for ch in target:
        new = [[0] * (m + 1) for _ in range(n + 1)]
        for b in range(m + 1):
            run = 0
            for a in range(n + 1):
                run += cur[a][b]
                if a < n and word1[a] == ch:
                    new[a][b] += run             # SAI: phải là new[a + 1][b]
        for a in range(n + 1):
            run = 0
            for b in range(m + 1):
                run += cur[a][b]
                if b < m and word2[b] == ch:
                    new[a][b + 1] += run
        cur = new
    return sum(cur[a][b] for a in range(1, n + 1) for b in range(1, m + 1)) % MOD

print("sai :", wrong_reuse("cd", "cd", "ccd"), " đúng:", Solution().interleaveCharacters("cd", "cd", "ccd"))

sai : 2  đúng: 4


### Lỗi 3 — Gộp hai cách chọn thành một (đếm "tập ký tự" thay vì đếm lựa chọn)

**Code sai:** dùng `set` các trạng thái hoặc coi hai nguồn cùng chữ cái là một.
**Vì sao sai:** đề nói hai cách khác nhau nếu **nguồn hoặc chỉ số** khác ở một vị trí. Ví dụ 2 đếm `word1[0]='c', word2[0]='c'` khác với `word2[0]='c', word1[0]='c'` (thứ tự theo `target` khác nhau).
**Code đúng:** để DP cộng dồn (số), mỗi nhánh một lần, không khử trùng bằng `set`.

### Lỗi 4 — Quên mod / trừ xong ra số âm ở cách bù trừ

**Code sai:** `return total - e1 - e2` (có thể âm sau khi đã `mod` từng vế).
**Code đúng:** `return (total - e1 - e2) % MOD`.

In [21]:
total, e1, e2 = 3, 1, 5
print("sai :", total - e1 - e2, "  đúng:", (total - e1 - e2) % MOD)

sai : -3   đúng: 1000000004


## 14. Mẹo & nhận dạng pattern

- **"Đếm số cách chọn dãy con" + "hai nguồn"** → thêm một chiều trạng thái cho mỗi nguồn.
- Nếu đề nói "ít nhất một ... từ mỗi ..." hãy xem có mã hóa trạng thái "chưa dùng" (giá trị đặc biệt) để cờ nằm sẵn trong trạng thái, hoặc dùng **bù trừ**: tổng − (thiếu A) − (thiếu B) + (thiếu cả hai).
- Khi bước chuyển có dạng "cộng mọi trạng thái cũ có chỉ số `<=` x" → nghĩ ngay **tổng tiền tố**.
- Giới hạn 100 cho ba đại lượng → `10^6` trạng thái là vừa phải; chỉ cần mỗi ô `O(1)`.
- Với DP lớp theo `k`, hãy **cuộn mảng** (chỉ giữ `cur`, `new`) để tiết kiệm bộ nhớ.
- Kiểm tra độc lập bằng một công thức khác (cách 4) giúp bắt lỗi chỉ số lệch một đơn vị.

## 15. Biến thể

1. **Không bắt buộc dùng cả hai chuỗi:** đáp án là tổng mọi trạng thái; chính là `g[T][n][m]` ở cách 4.
2. **Ba chuỗi nguồn:** thêm chiều thứ ba, DP `O(T·n·m·p)`; điều kiện "dùng đủ cả ba" xử lý bằng trạng thái "chưa dùng" cho từng nguồn.
3. **Cho phép chỉ số không tăng ngặt (tăng không ngặt):** đổi `i >= a` thành `i >= a - 1` (được dùng lại cùng chỉ số).
4. **Đếm khác nhau theo chuỗi kết quả:** như bài *Distinct Subsequences II*, phải khử trùng theo giá trị chứ không theo chỉ số.
5. **Thêm ràng buộc xen kẽ** (không được lấy hai ký tự liên tiếp cùng một nguồn): thêm một chiều "nguồn của ký tự trước".

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi (nguồn, chỉ số) cho từng ký tự | mũ | `O(T)` | chỉ để kiểm chứng input rất nhỏ |
| 2. DP chuyển `O(n+m)` | trạng thái (a, b) và duyệt chỉ số kế tiếp | `O(T·n·m·(n+m))` | `O(n·m)` | hiểu bài, input nhỏ |
| 3. DP tổng tiền tố | gộp vòng lặp lồng bằng `run` | `O(T·n·m)` | `O(n·m)` | **nộp bài** |
| 4. Tiền tố + bù trừ | đếm tất cả rồi trừ "thiếu một nguồn" | `O(T·n·m)` | `O(n·m)` | đối chiếu, biến thể "không bắt buộc cả hai" |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `count_embeddings("abab", "ab")` bằng bao nhiêu? (Đếm số cách chọn chỉ số tăng ngặt.)

**Bài 2 (tính tay).** `word1 = "aa"`, `word2 = "a"`, `target = "aa"`. Có bao nhiêu cách? (Gợi ý: liệt kê `S1`, `S2`.)

**Bài 3 (điền TODO).** Hoàn thiện `my_count_embeddings(word, target)` — DP một chiều, duyệt `t` từ cao xuống thấp.

**Bài 4 (tự viết lại).** Viết `my_count_ways(word1, word2, target)` theo cách 3 mà không nhìn code nộp.

**Bài 5 (biến thể).** Viết `ways_at_least_word1(word1, word2, target)`: số cách (không bắt buộc dùng `word2`) nhưng phải dùng **ít nhất một ký tự của `word1`**.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_count_embeddings(word, target):
    # TODO 1: f = [1] + [0]*len(target)
    # TODO 2: với mỗi ký tự c của word:
    # TODO 3:     với t từ len(target) xuống 1: nếu target[t-1] == c thì f[t] += f[t-1]
    # TODO 4: trả về f[len(target)] % MOD
    pass

# Bài 4 — TODO
def my_count_ways(word1, word2, target):
    # TODO: cur[a][b], quét tổng tiền tố theo a (word1) và theo b (word2), cộng a>=1 và b>=1
    pass

# Bài 5 — TODO
def ways_at_least_word1(word1, word2, target):
    # TODO: tổng mọi trạng thái trừ đi trạng thái a == 0 (không dùng word1)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê các chỉ số của `a` và `b` trong `"abab"`.
- Mức 2: `a` ở `0, 2`; `b` ở `1, 3`; cần `a` đứng trước `b`.
- Mức 3: cặp hợp lệ `(0,1)`, `(0,3)`, `(2,3)` → `3`.

**Bài 2.**
- Mức 1: mỗi ký tự `a` của `target` được gán nguồn; cần cả hai nguồn dùng ít nhất một lần.
- Mức 2: `S1 = {0}`, `S2 = {1}` → `word1` chứa `"a"` 2 cách × `word2` chứa `"a"` 1 cách = 2; `S1 = {1}`, `S2 = {0}` → 2 cách.
- Mức 3: `S1 = {0,1}` cho `S2` rỗng bị loại. Tổng = `2 + 2 = 4`.

**Bài 3.**
- Mức 1: duyệt `t` ngược để không dùng một ký tự `c` hai lần.
- Mức 2: `f[t]` = số cách khớp `t` ký tự đầu của `target`.
- Mức 3: `if target[t-1] == c: f[t] += f[t-1]`.

**Bài 4.**
- Mức 1: bảng `(n+1) × (m+1)`, `cur[0][0] = 1`.
- Mức 2: với mỗi ký tự `ch`, hai lượt quét có `run`.
- Mức 3: `new[a+1][b] += run` khi `word1[a] == ch`.

**Bài 5.**
- Mức 1: không bắt buộc `word2` nghĩa là `b` có thể bằng `0`.
- Mức 2: vẫn cần `a >= 1`.
- Mức 3: cộng `cur[a][b]` với `a >= 1` và mọi `b`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert count_embeddings("abab", "ab") == 3
# Bài 2
assert Solution().interleaveCharacters("aa", "a", "aa") == 4

# Bài 3
def my_count_embeddings_ans(word, target):
    f = [1] + [0] * len(target)
    for c in word:
        for t in range(len(target), 0, -1):
            if target[t - 1] == c:
                f[t] += f[t - 1]
    return f[len(target)] % MOD

for w, t in [("abab", "ab"), ("aaaa", "aa"), ("abc", "d"), ("rabbbit", "rabbit")]:
    assert my_count_embeddings_ans(w, t) == count_embeddings(w, t)
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_count_ways_ans(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1
    for ch in target:
        new = [[0] * (m + 1) for _ in range(n + 1)]
        for b in range(m + 1):
            run = 0
            for a in range(n + 1):
                run += cur[a][b]
                if a < n and word1[a] == ch:
                    new[a + 1][b] = (new[a + 1][b] + run) % MOD
        for a in range(n + 1):
            run = 0
            for b in range(m + 1):
                run += cur[a][b]
                if b < m and word2[b] == ch:
                    new[a][b + 1] = (new[a][b + 1] + run) % MOD
        cur = new
    return sum(cur[a][b] for a in range(1, n + 1) for b in range(1, m + 1)) % MOD

# Bài 5: số cách dùng ít nhất một ký tự của word1 (word2 tùy ý)
def ways_at_least_word1_ans(word1, word2, target):
    n, m = len(word1), len(word2)
    cur = [[0] * (m + 1) for _ in range(n + 1)]
    cur[0][0] = 1
    for ch in target:
        new = [[0] * (m + 1) for _ in range(n + 1)]
        for b in range(m + 1):
            run = 0
            for a in range(n + 1):
                run += cur[a][b]
                if a < n and word1[a] == ch:
                    new[a + 1][b] = (new[a + 1][b] + run) % MOD
        for a in range(n + 1):
            run = 0
            for b in range(m + 1):
                run += cur[a][b]
                if b < m and word2[b] == ch:
                    new[a][b + 1] = (new[a][b + 1] + run) % MOD
        cur = new
    return sum(cur[a][b] for a in range(1, n + 1) for b in range(m + 1)) % MOD

def brute_at_least_word1(word1, word2, target):
    n, m, T = len(word1), len(word2), len(target)
    def go(k, p1, p2, used1):
        if k == T:
            return 1 if used1 else 0
        ch, tot = target[k], 0
        for i in range(p1, n):
            if word1[i] == ch:
                tot += go(k + 1, i + 1, p2, True)
        for j in range(p2, m):
            if word2[j] == ch:
                tot += go(k + 1, p1, j + 1, used1)
        return tot
    return go(0, 0, 0, False) % MOD

rng = random.Random(5)
for _ in range(300):
    w1 = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    w2 = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    t  = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    assert my_count_ways_ans(w1, w2, t) == Solution().interleaveCharacters(w1, w2, t)
    assert ways_at_least_word1_ans(w1, w2, t) == brute_at_least_word1(w1, w2, t)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP "hai nguồn, tổng tiền tố":**

```python
MOD = 10**9 + 7
cur = [[0] * (m + 1) for _ in range(n + 1)]
cur[0][0] = 1                                  # a=0: chưa dùng word1, b=0: chưa dùng word2
for ch in target:
    new = [[0] * (m + 1) for _ in range(n + 1)]
    for b in range(m + 1):                     # lấy từ word1
        run = 0
        for a in range(n + 1):
            run += cur[a][b]
            if a < n and word1[a] == ch:
                new[a + 1][b] = (new[a + 1][b] + run) % MOD
    for a in range(n + 1):                     # lấy từ word2
        run = 0
        for b in range(m + 1):
            run += cur[a][b]
            if b < m and word2[b] == ch:
                new[a][b + 1] = (new[a][b + 1] + run) % MOD
    cur = new
return sum(cur[a][b] for a in range(1, n + 1) for b in range(1, m + 1)) % MOD
```

**Checklist trước khi submit:**

- [ ] Mã hóa `a = 0` (chưa dùng) và `a = i + 1` (đã dùng tới chỉ số `i`).
- [ ] Trạng thái mới là `(i + 1, b)` hoặc `(a, j + 1)` — chỉ số tăng ngặt.
- [ ] Cộng `run` **trước** khi so sánh `word1[a] == ch` (tổng gồm cả `a`).
- [ ] Hai lượt (word1, word2) đều đọc từ `cur`, ghi vào `new`.
- [ ] Đáp án chỉ lấy `a >= 1` và `b >= 1`.
- [ ] `% MOD` sau mỗi lần cộng vào `new`; với phép trừ thì `% MOD` cuối cùng.
- [ ] Đối chiếu vét cạn trên input nhỏ ngẫu nhiên trước khi nộp.